In [1]:
import requests
import pandas as pd

url = "https://air-quality-api.open-meteo.com/v1/air-quality"
params = {
    "latitude": 10.85,
    "longitude": 106.75,
    "hourly": "pm2_5,pm10",
    "start_date": "2021-09-01",
    "end_date": "2025-08-31",
    "timezone": "Asia/Ho_Chi_Minh"
}

r = requests.get(url, params=params)
data = r.json()
print(data.keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])


In [2]:
df = pd.DataFrame(data['hourly'])
df['time'] = pd.to_datetime(df['time'])
df

,time,pm2_5,pm10
0,2021-09-01 00:00:00,NaN,NaN
1,2021-09-01 01:00:00,NaN,NaN
2,2021-09-01 02:00:00,NaN,NaN
3,2021-09-01 03:00:00,NaN,NaN
4,2021-09-01 04:00:00,NaN,NaN
...,...,...,...
35059,2025-08-31 19:00:00,27.1,27.6
35060,2025-08-31 20:00:00,30.2,31.5
35061,2025-08-31 21:00:00,32.9,34.6
35062,2025-08-31 22:00:00,33.8,35.2


In [3]:
df['meteo_pm2_5_next1'] = df['pm2_5'].shift(-1)
df['meteo_pm2_5_next2'] = df['pm2_5'].shift(-2)
df['meteo_pm2_5_next3'] = df['pm2_5'].shift(-3)
df['meteo_pm10_next1'] = df['pm10'].shift(-1)
df['meteo_pm10_next2'] = df['pm10'].shift(-2)
df['meteo_pm10_next3'] = df['pm10'].shift(-3)
df = df[:-3]
df

,time,pm2_5,pm10,meteo_pm2_5_next1,meteo_pm2_5_next2,meteo_pm2_5_next3,meteo_pm10_next1,meteo_pm10_next2,meteo_pm10_next3
0,2021-09-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-09-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-09-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-09-01 03:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-09-01 04:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
35056,2025-08-31 16:00:00,20.4,20.7,28.6,33.8,27.1,28.9,34.1,27.6
35057,2025-08-31 17:00:00,28.6,28.9,33.8,27.1,30.2,34.1,27.6,31.5
35058,2025-08-31 18:00:00,33.8,34.1,27.1,30.2,32.9,27.6,31.5,34.6
35059,2025-08-31 19:00:00,27.1,27.6,30.2,32.9,33.8,31.5,34.6,35.2


In [4]:
baseline_val = df[(df["time"] >= pd.to_datetime("2024-09-01")) & (df["time"] < pd.to_datetime("2025-03-01"))]
baseline_test = df[df["time"] >= pd.to_datetime("2025-03-01")]

In [6]:
ourdata = pd.read_csv(r'../3. Data Preprocessing/Data_Ready.csv')
ourdata['time'] = pd.to_datetime(ourdata['time'], errors='coerce')
ourdata_val = ourdata[(ourdata["time"] >= pd.to_datetime("2024-09-01")) & (ourdata["time"] < pd.to_datetime("2025-03-01"))]
ourdata_test = ourdata[ourdata["time"] >= pd.to_datetime("2025-03-01")]

In [7]:
print(len(baseline_val), len(ourdata_val))
print(len(baseline_test), len(ourdata_test))

4344 4344
4413 4413


In [8]:
import numpy as np
rmse_pm2_5_next1 = np.sqrt(((ourdata_test['pm2_5_next1'] - baseline_test['meteo_pm2_5_next1'])**2).mean())
rmse_pm2_5_next2 = np.sqrt(((ourdata_test['pm2_5_next2'] - baseline_test['meteo_pm2_5_next2'])**2).mean())
rmse_pm2_5_next3 = np.sqrt(((ourdata_test['pm2_5_next3'] - baseline_test['meteo_pm2_5_next3'])**2).mean())
rmse_pm10_next1  = np.sqrt(((ourdata_test['pm10_next1'] - baseline_test['meteo_pm10_next1'])**2).mean())
rmse_pm10_next2  = np.sqrt(((ourdata_test['pm10_next2'] - baseline_test['meteo_pm10_next2'])**2).mean())
rmse_pm10_next3  = np.sqrt(((ourdata_test['pm10_next3'] - baseline_test['meteo_pm10_next3'])**2).mean())

In [9]:
print("RMSE PM2.5 Next1:", rmse_pm2_5_next1)
print("RMSE PM2.5 Next2:", rmse_pm2_5_next2)
print("RMSE PM2.5 Next3:", rmse_pm2_5_next3)
print("RMSE PM10 Next1:", rmse_pm10_next1)  
print("RMSE PM10 Next2:", rmse_pm10_next2)
print("RMSE PM10 Next3:", rmse_pm10_next3)

RMSE PM2.5 Next1: 29.045115774426502
RMSE PM2.5 Next2: 29.049011680726334
RMSE PM2.5 Next3: 29.052181149574018
RMSE PM10 Next1: 34.50856079111486
RMSE PM10 Next2: 34.51161816423101
RMSE PM10 Next3: 34.514019513385804
